# Projeto 2: Logística - Análise de Atrasos

Performance de entrega e identificação de gargalos

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

## 2.1 Tabela: ft_atrasos_pedidos_local_vendedor

In [0]:
df_pedidos = spark.table("silver.ft_pedidos")
df_consumidores = spark.table("silver.ft_consumidores")
df_itens = spark.table("silver.ft_itens_pedidos")

df_atrasos = df_pedidos.alias("p") \
    .join(df_consumidores.alias("c"), col("p.id_consumidor") == col("c.id_consumidor")) \
    .join(df_itens.select("id_pedido", "id_vendedor").distinct().alias("i"), 
          col("p.id_pedido") == col("i.id_pedido")) \
    .select(
        col("p.id_pedido"),
        col("i.id_vendedor"),
        col("p.id_consumidor"),
        col("p.entrega_no_prazo"),
        col("p.tempo_entrega_dias").cast("int"),
        col("p.tempo_entrega_estimado_dias").cast("int"),
        col("c.cidade"),
        col("c.estado")
    )

df_atrasos.write.format("delta").mode("overwrite").saveAsTable("gold.ft_atrasos_pedidos_local_vendedor")

In [0]:
print(f"Registros criados: {spark.table('gold.ft_atrasos_pedidos_local_vendedor').count()}")
spark.table("gold.ft_atrasos_pedidos_local_vendedor").show(5)

## 2.2 Views Analíticas

### 2.2.1 View: view_tempo_medio_entrega_localidade

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW gold.view_tempo_medio_entrega_localidade AS
    SELECT 
        cidade,
        estado,
        CAST(AVG(tempo_entrega_dias) AS DECIMAL(10,2)) AS tempo_medio_entrega,
        CAST(AVG(tempo_entrega_estimado_dias) AS DECIMAL(10,2)) AS tempo_medio_estimado,
        CASE WHEN AVG(tempo_entrega_dias) > AVG(tempo_entrega_estimado_dias) 
             THEN 'SIM' ELSE 'NÃO' END AS entrega_maior_que_estimado
    FROM gold.ft_atrasos_pedidos_local_vendedor
    WHERE tempo_entrega_dias IS NOT NULL AND tempo_entrega_estimado_dias IS NOT NULL
    GROUP BY cidade, estado
    ORDER BY tempo_medio_entrega DESC
""")

In [0]:
spark.sql("SELECT * FROM gold.view_tempo_medio_entrega_localidade LIMIT 10").show()

### 2.2.2 View: view_vendedor_pontualidade

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW gold.view_vendedor_pontualidade AS
    SELECT 
        id_vendedor,
        COUNT(DISTINCT id_pedido) AS total_pedidos,
        SUM(CASE WHEN entrega_no_prazo = 'Não' THEN 1 ELSE 0 END) AS total_atrasados,
        CAST(SUM(CASE WHEN entrega_no_prazo = 'Não' THEN 1 ELSE 0 END) * 100.0 / 
             COUNT(DISTINCT id_pedido) AS DECIMAL(5,2)) AS percentual_atraso
    FROM gold.ft_atrasos_pedidos_local_vendedor
    GROUP BY id_vendedor
    ORDER BY percentual_atraso DESC
""")

In [0]:
spark.sql("SELECT * FROM gold.view_vendedor_pontualidade LIMIT 10").show()

## Validação

In [0]:
print("Objetos criados neste notebook:")
spark.sql("SHOW TABLES IN gold").filter(
    "tableName LIKE '%atrasos%' OR tableName LIKE '%entrega%' OR tableName LIKE '%pontualidade%'"
).show(truncate=False)